In [18]:
import pandas as pd
import numpy as np

df = pd.read_csv('2021_2023_시계열_지하수_기상_test_inputs.csv', encoding='cp949')

In [5]:
df = df.sort_values(by=['code_new', 'ymd'])

In [19]:
df.info

<bound method DataFrame.info of                      ymd  code_new  wtemp   ec  기온(°C)  강수량(mm)  풍속(m/s)  \
0        2021-01-01 0:00         1   16.8  897    -4.5      0.8      1.2   
1        2021-01-01 1:00         1   16.8  897    -5.9      0.0      0.8   
2        2021-01-01 2:00         1   16.8  897    -7.2      0.0      1.5   
3        2021-01-01 3:00         1   16.8  897    -7.5      0.0      1.3   
4        2021-01-01 4:00         1   16.8  897    -8.0      0.0      0.1   
...                  ...       ...    ...  ...     ...      ...      ...   
305318  2023-12-31 19:00        12   15.0  450     3.8      0.0      1.2   
305319  2023-12-31 20:00        12   15.0  450     3.6      0.0      0.4   
305320  2023-12-31 21:00        12   15.0  450     3.5      0.0      0.4   
305321  2023-12-31 22:00        12   15.0  450     3.4      0.0      0.6   
305322  2023-12-31 23:00        12   15.0  450     3.4      0.0      0.5   

        습도(%)  현지기압(hPa)  지면온도(°C)  
0        75.0     

In [12]:
site_col = 'code_new'

for i in range(1, 13):
    df[df[site_col] == i].to_csv(f'{i}.csv', index=False, encoding='cp949')

여기부터 하세요

In [99]:
df = pd.read_csv('12.csv', encoding='cp949')

In [100]:
df.info

<bound method DataFrame.info of                     ymd  code_new    elev  wtemp   ec  기온(°C)  강수량(mm)  \
0         2014.1.1 0:00        12  113.20   15.0  377     3.8      0.0   
1         2014.1.1 1:00        12  113.20   15.0  378     3.6      0.0   
2         2014.1.1 2:00        12  113.20   15.0  377     3.3      0.0   
3         2014.1.1 3:00        12  113.20   15.0  377     2.5      0.0   
4         2014.1.1 4:00        12  113.20   15.0  377     0.9      0.0   
...                 ...       ...     ...    ...  ...     ...      ...   
60735  2020.12.31 19:00        12  113.41   14.8  449    -2.7      0.0   
60736  2020.12.31 20:00        12  113.41   14.8  449    -3.5      0.0   
60737  2020.12.31 21:00        12  113.41   14.8  449    -2.7      1.4   
60738  2020.12.31 22:00        12  113.41   14.8  449    -3.3      0.0   
60739  2020.12.31 23:00        12  113.40   14.8  449    -4.9      0.0   

       풍속(m/s)  습도(%)  현지기압(hPa)  지면온도(°C)  rain_event_3h  rain_event_12h  
0  

In [101]:
# 날짜 처리
df['ymd'] = pd.to_datetime(df['ymd'])
df = df.sort_values('ymd').reset_index(drop=True)
df = df.set_index('ymd')

# 주요 컬럼 지정
rain_col = '강수량(mm)'
gw_col   = 'elev'
temp_col = '기온(°C)'
pres_col = '현지기압(hPa)'

# 1) 3·12·24·48·168시간 누적 강수량
# -> 소수점 부동소수 오차 방지를 위해 '×10 정수화 → rolling sum → ÷10 → 1자리 반올림'
site_col = 'code_new'
for w in [3, 12, 24, 48, 168]:
    df[f'sum_of_rain_in_{w}h'] = (
        df.groupby(site_col)[rain_col]
          .apply(lambda s: (
              (s * 10).round().astype('Int64')              # 0.1mm 단위를 정수(십분의일 mm)로
                .rolling(window=w, min_periods=w).sum()      # 정수 합
                .astype(float).div(10).round(1)              # 다시 mm로 환산 + 1자리 반올림
          ))
          .reset_index(level=0, drop=True)
    )

# 2) 최근 k시간 최대 1시간 강수
for w in [3, 6, 12, 24]:
    df[f'max_rain_in_{w}h'] = df[rain_col].rolling(window=w, min_periods=1).max()

# 3) 최근 비 이후 경과 시간
is_rain = (df[rain_col] > 0).astype(int)

last_rain_time = pd.Series(np.where(is_rain==1, df.index.view('int64'), np.nan), index=df.index)
last_rain_time = last_rain_time.ffill()
df['hours_since_rain'] = (df.index.view('int64') - last_rain_time) / 1e9 / 3600.0

# 4) API (half-life 24h, 72h)
def api_series(rain, half_life_hours):
    alpha = 1 - 0.5**(1/half_life_hours)
    out = []
    acc = 0
    for r in rain:
        acc = alpha*r + (1-alpha)*acc
        out.append(acc)
    return out

df['API_hl_24h'] = api_series(df[rain_col].fillna(0), 24)
df['API_hl_72h'] = api_series(df[rain_col].fillna(0), 72)

# 5) 강수 lag
for lag in [1, 3, 6, 12, 24, 48, 72]:
    df[f'rain_lag_{lag}h'] = df[rain_col].shift(lag)

# 6) 지하수 자기회귀
for lag in [168]:
    df[f'elev_lag_{lag}h'] = df[gw_col].shift(lag)

# 7) 계절성 인코딩
hour = df.index.hour
dayofyear = df.index.dayofyear
df['hour_sin'] = np.sin(2*np.pi*hour/24)
df['hour_cos'] = np.cos(2*np.pi*hour/24)
df['day_sin']  = np.sin(2*np.pi*dayofyear/365.25)
df['day_cos']  = np.cos(2*np.pi*dayofyear/365.25)

# 8) 기상 변수 보강 (예: 기온, 기압)
for col in [temp_col, pres_col]:
    for w in [6, 24]:
        df[f'{col}_mean_{w}h'] = df[col].rolling(window=w, min_periods=1).mean()
        df[f'{col}_std_{w}h']  = df[col].rolling(window=w, min_periods=1).std()
    df[f'Δ{col}_1h'] = df[col].diff(1)

In [102]:
df.info

<bound method DataFrame.info of                      code_new    elev  wtemp   ec  기온(°C)  강수량(mm)  풍속(m/s)  \
ymd                                                                           
2014-01-01 00:00:00        12  113.20   15.0  377     3.8      0.0      2.6   
2014-01-01 01:00:00        12  113.20   15.0  378     3.6      0.0      2.4   
2014-01-01 02:00:00        12  113.20   15.0  377     3.3      0.0      2.2   
2014-01-01 03:00:00        12  113.20   15.0  377     2.5      0.0      1.0   
2014-01-01 04:00:00        12  113.20   15.0  377     0.9      0.0      1.4   
...                       ...     ...    ...  ...     ...      ...      ...   
2020-12-31 19:00:00        12  113.41   14.8  449    -2.7      0.0      1.1   
2020-12-31 20:00:00        12  113.41   14.8  449    -3.5      0.0      0.7   
2020-12-31 21:00:00        12  113.41   14.8  449    -2.7      1.4      2.3   
2020-12-31 22:00:00        12  113.41   14.8  449    -3.3      0.0      2.0   
2020-12-31 23:00:00 

In [103]:
# 결과 저장
df.to_csv("12_after.csv", encoding="utf-8-sig")
print("엑셀 파일 저장 완료")

엑셀 파일 저장 완료


In [105]:
files = [f"{i}_after.csv" for i in range(1, 13)]
dfs = [pd.read_csv(fp, encoding="utf-8") for fp in files]
merged = pd.concat(dfs, ignore_index=True, sort=False)

# 필요하면 변수명 df로 사용
df = merged

# 저장
merged.to_csv("after_merged.csv", index=False, encoding="cp949")
print('완')

완


여기까진 ㅈㄴ 쓸데없는 추가 자료 만들었던 부분

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('2014_2020_train_interpolated_small.csv', encoding='cp949')

In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 305323 entries, 0 to 305322
Data columns (total 10 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   ymd        305323 non-null  object 
 1   code_new   305323 non-null  int64  
 2   wtemp      305323 non-null  float64
 3   ec         305323 non-null  int64  
 4   기온(°C)     305159 non-null  float64
 5   강수량(mm)    305323 non-null  float64
 6   풍속(m/s)    304904 non-null  float64
 7   습도(%)      305040 non-null  float64
 8   현지기압(hPa)  305166 non-null  float64
 9   지면온도(°C)   303971 non-null  float64
dtypes: float64(7), int64(2), object(1)
memory usage: 23.3+ MB


In [16]:
# 필수 컬럼 확인
required = ["code_new", "ymd"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"필수 컬럼 누락: {missing}. 현재 컬럼: {df.columns.tolist()}")

# ymd를 datetime으로 변환
df["ymd"] = pd.to_datetime(df["ymd"], errors="coerce")
# if df["ymd"].isna().any():
#     print("⚠️ 변환 불가 타임스탬프 예시:")
#     display(df[df["ymd"].isna()].head())
#     raise ValueError("ymd에 변환 불가 값이 있습니다. 위 예시 확인 후 원본 데이터 수정 필요")

# 정렬 (지점 → 시간)
df = df.sort_values(["code_new", "ymd"]).reset_index(drop=True)

# 전체 시간 인덱스 (2014-01-01 00:00 ~ 2020-12-31 23:00, 1시간 간격)
full_idx = pd.date_range("2021-01-01 00:00:00", "2023-12-31 23:00:00", freq="H")
len(full_idx), full_idx[0], full_idx[-1]

C:\Users\jcu03\AppData\Local\Temp\ipykernel_25468\2000891742.py:18: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  full_idx = pd.date_range("2021-01-01 00:00:00", "2023-12-31 23:00:00", freq="H")


(26280, Timestamp('2021-01-01 00:00:00'), Timestamp('2023-12-31 23:00:00'))

In [17]:
# 메모리/속도 안정성을 위해 groupby 루프 방식 권장
filled_chunks = []
inserted_counts = []

# 컬럼 순서 보존용
col_order = df.columns.tolist()
if "inserted" not in col_order:
    col_order = col_order + ["inserted"]

for code, g in df.groupby("code_new", sort=False):
    g2 = g.set_index("ymd")
    # full_idx로 재인덱싱: 빠진 시간은 NaN 행으로 생김
    g_re = g2.reindex(full_idx)
    # 지점 코드 채우기 (NaN 행에도)
    g_re["code_new"] = code
    # 어떤 행이 새로 추가되었는지 표시
    g_re["inserted"] = ~g_re.index.isin(g2.index)
    inserted_counts.append({"code_new": code, "missing_rows_added": int(g_re["inserted"].sum())})
    filled_chunks.append(g_re)

# 합치고 컬럼/정렬 정리
df_full = pd.concat(filled_chunks).reset_index().rename(columns={"index": "ymd"})
df_full = df_full.sort_values(["code_new", "ymd"]).reset_index(drop=True)
# 컬럼 순서 맞추기 (없던 컬럼이 있다면 뒤에 붙음)
df_full = df_full[[c for c in col_order if c in df_full.columns] + [c for c in df_full.columns if c not in col_order]]

# 요약 확인
summary = pd.DataFrame(inserted_counts).sort_values("code_new").reset_index(drop=True)
display(summary.head(12))
df_full.head()

,code_new,missing_rows_added
0,1,0
1,2,0
2,3,0
3,4,0
4,5,0
5,6,0
6,7,0
7,8,0
8,9,0
9,10,0


,ymd,code_new,wtemp,ec,기온(°C),강수량(mm),풍속(m/s),습도(%),현지기압(hPa),지면온도(°C),inserted,is_precipitation_missing
0,2021-01-01 00:00:00,1,16.8,897.0,-4.5,0.8,1.2,75.0,1011.5,0.3,False,0
1,2021-01-01 01:00:00,1,16.8,897.0,-5.9,0.0,0.8,79.0,1011.0,0.2,False,0
2,2021-01-01 02:00:00,1,16.8,897.0,-7.2,0.0,1.5,88.0,1010.7,0.2,False,0
3,2021-01-01 03:00:00,1,16.8,897.0,-7.5,0.0,1.3,90.0,1011.1,0.2,False,0
4,2021-01-01 04:00:00,1,16.8,897.0,-8.0,0.0,0.1,91.0,1011.3,0.1,False,0


In [10]:
out_path = "2021_2023_시계열_지하수_기상_test_inputs_interpolated.csv"
df_full.to_csv(out_path, index=False, encoding="cp949")
out_path

'2021_2023_시계열_지하수_기상_test_inputs_interpolated.csv'

In [11]:
df = pd.read_csv('2021_2023_시계열_지하수_기상_test_inputs_interpolated.csv', encoding='cp949')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 315360 entries, 0 to 315359
Data columns (total 11 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   ymd        315360 non-null  object 
 1   code_new   315360 non-null  int64  
 2   wtemp      305323 non-null  float64
 3   ec         305323 non-null  float64
 4   기온(°C)     305159 non-null  float64
 5   강수량(mm)    305323 non-null  float64
 6   풍속(m/s)    304904 non-null  float64
 7   습도(%)      305040 non-null  float64
 8   현지기압(hPa)  305166 non-null  float64
 9   지면온도(°C)   303971 non-null  float64
 10  inserted   315360 non-null  bool   
dtypes: bool(1), float64(8), int64(1), object(1)
memory usage: 24.4+ MB


In [12]:
# 0) 기본 정리: 시간형 변환 + 정렬
df["ymd"] = pd.to_datetime(df["ymd"], errors="raise")
df = df.sort_values(["code_new", "ymd"]).reset_index(drop=True)

# 1) 선형 보간 대상 컬럼 (강수량 제외)
interp_cols = ["elev", "wtemp", "ec", "기온(°C)", "풍속(m/s)", "습도(%)", "현지기압(hPa)", "지면온도(°C)"]

# 2) inserted==True 행만 선형 보간으로 채우기 (지점별·시간순)
def _interp_group(g):
    g = g.set_index("ymd")
    for col in interp_cols:
        if col not in g.columns:
            continue
        # 시간기반 선형보간 (양쪽 값이 있을 때만 채워짐)
        s_interp = g[col].interpolate(method="time")
        # 원본 값은 보존, inserted==True 인 곳만 대체
        mask = g["inserted"] & g[col].isna()
        g.loc[mask, col] = s_interp[mask]
    return g.reset_index()

df = (df
      .groupby("code_new", group_keys=False)
      .apply(_interp_group)
      .sort_values(["code_new", "ymd"])
      .reset_index(drop=True))

# 3) 강수량(mm): 0으로 채우되, 결측에서 채웠음을 알려주는 플래그 추가
rain_col = "강수량(mm)"
flag_col = "is_precipitation_missing"

# inserted==True & 강수량 결측이었던 행만 마킹
mask_missing_rain_at_inserted = df["inserted"] & df[rain_col].isna()

# 플래그 초기화(0) 후, 해당 마스크에만 1
df[flag_col] = 0
df.loc[mask_missing_rain_at_inserted, flag_col] = 1

# 그다음, 그 마스크에 한해서 강수량을 0으로 대체
df.loc[mask_missing_rain_at_inserted, rain_col] = 0.0

# 4) 확인용(필요하면 주석 해제)
display(df.query("inserted").head(20))
df.info()

C:\Users\jcu03\AppData\Local\Temp\ipykernel_25468\3512569085.py:23: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_interp_group)


,ymd,code_new,wtemp,ec,기온(°C),강수량(mm),풍속(m/s),습도(%),현지기압(hPa),지면온도(°C),inserted,is_precipitation_missing
3133,2021-05-11 13:00:00,1,14.7,706.5,19.776923,0.0,4.388462,54.576923,997.969231,26.680769,True,1
3134,2021-05-11 14:00:00,1,14.7,705.0,20.053846,0.0,4.376923,54.153846,997.838462,27.061538,True,1
3135,2021-05-11 15:00:00,1,14.7,703.5,20.330769,0.0,4.365385,53.730769,997.707692,27.442308,True,1
3136,2021-05-11 16:00:00,1,14.7,702.0,20.607692,0.0,4.353846,53.307692,997.576923,27.823077,True,1
3137,2021-05-11 17:00:00,1,14.7,700.5,20.884615,0.0,4.342308,52.884615,997.446154,28.203846,True,1
3138,2021-05-11 18:00:00,1,14.7,699.0,21.161538,0.0,4.330769,52.461538,997.315385,28.584615,True,1
3139,2021-05-11 19:00:00,1,14.7,697.5,21.438462,0.0,4.319231,52.038462,997.184615,28.965385,True,1
3140,2021-05-11 20:00:00,1,14.7,696.0,21.715385,0.0,4.307692,51.615385,997.053846,29.346154,True,1
3141,2021-05-11 21:00:00,1,14.7,694.5,21.992308,0.0,4.296154,51.192308,996.923077,29.726923,True,1
3142,2021-05-11 22:00:00,1,14.7,693.0,22.269231,0.0,4.284615,50.769231,996.792308,30.107692,True,1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 315360 entries, 0 to 315359
Data columns (total 12 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   ymd                       315360 non-null  datetime64[ns]
 1   code_new                  315360 non-null  int64         
 2   wtemp                     315360 non-null  float64       
 3   ec                        315360 non-null  float64       
 4   기온(°C)                    315196 non-null  float64       
 5   강수량(mm)                   315360 non-null  float64       
 6   풍속(m/s)                   314941 non-null  float64       
 7   습도(%)                     315077 non-null  float64       
 8   현지기압(hPa)                 315203 non-null  float64       
 9   지면온도(°C)                  314008 non-null  float64       
 10  inserted                  315360 non-null  bool          
 11  is_precipitation_missing  315360 non-null  int64         
dtypes:

In [33]:
df.to_csv("2014_2020_train_full_interpolated_filled.csv", index=False, float_format="%.5f", encoding="cp949")

In [7]:
df = pd.read_csv('2014_2020_train_full_interpolated_filled.csv', encoding='cp949')

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 736416 entries, 0 to 736415
Data columns (total 13 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   ymd                       736416 non-null  object 
 1   code_new                  736416 non-null  int64  
 2   elev                      736416 non-null  float64
 3   wtemp                     736416 non-null  float64
 4   ec                        736416 non-null  float64
 5   기온(°C)                    736416 non-null  float64
 6   강수량(mm)                   736416 non-null  float64
 7   풍속(m/s)                   736416 non-null  float64
 8   습도(%)                     736416 non-null  float64
 9   현지기압(hPa)                 736416 non-null  float64
 10  지면온도(°C)                  736416 non-null  float64
 11  inserted                  736416 non-null  bool   
 12  is_precipitation_missing  736416 non-null  int64  
dtypes: bool(1), float64(9), int64(2), object(1)


In [9]:
site_col = 'code_new'

for i in range(1, 13):
    df[df[site_col] == i].to_csv(f'{i}.csv', index=False, encoding='cp949')

In [43]:
df = pd.read_csv('12.csv', encoding='cp949')

In [44]:
# 날짜 처리
df['ymd'] = pd.to_datetime(df['ymd'])
df = df.sort_values('ymd').reset_index(drop=True)
df = df.set_index('ymd')

gw_col = 'elev'

# 지하수 자기회귀
for lag in [168]:
    df[f'elev_lag_{lag}h'] = df[gw_col].shift(lag)

In [45]:
# 결과 저장
df.to_csv("12_after.csv", encoding="cp949")
print("엑셀 파일 저장 완료")

엑셀 파일 저장 완료


In [13]:
files = [f"{i}_after.csv" for i in range(1, 13)]
dfs = [pd.read_csv(fp, encoding="cp949") for fp in files]
merged = pd.concat(dfs, ignore_index=True, sort=False)

# 필요하면 변수명 df로 사용
df = merged

# 저장
merged.to_csv("after_merged.csv", index=False, encoding="cp949")
print('완')

FileNotFoundError: [Errno 2] No such file or directory: '1_after.csv'

In [14]:
merged.info()

NameError: name 'merged' is not defined

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('2021_2023_시계열_지하수_기상_test_inputs.csv', encoding='cp949')

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 305323 entries, 0 to 305322
Data columns (total 10 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   ymd        305323 non-null  object 
 1   code_new   305323 non-null  int64  
 2   wtemp      305302 non-null  float64
 3   ec         305323 non-null  float64
 4   기온(°C)     305118 non-null  float64
 5   강수량(mm)    305323 non-null  float64
 6   풍속(m/s)    304860 non-null  float64
 7   습도(%)      304999 non-null  float64
 8   현지기압(hPa)  305125 non-null  float64
 9   지면온도(°C)   303904 non-null  float64
dtypes: float64(8), int64(1), object(1)
memory usage: 23.3+ MB


In [5]:
# -------------------------------
# 2) 결측치 확인
# -------------------------------
print("열별 결측치 개수:")
print(df.isna().sum())

# -------------------------------
# 3) 선형 보간 (세로 방향, 즉 열 기준)
# -------------------------------
df_interpolated = df.interpolate(method='linear', axis=0).round(1)

# -------------------------------
# 4) 보간 후 결측치 확인
# -------------------------------
print("\n보간 후 결측치 개수:")
print(df_interpolated.isna().sum())

열별 결측치 개수:
ymd             0
code_new        0
wtemp          21
ec              0
기온(°C)        205
강수량(mm)         0
풍속(m/s)       463
습도(%)         324
현지기압(hPa)     198
지면온도(°C)     1419
dtype: int64

보간 후 결측치 개수:
ymd          0
code_new     0
wtemp        0
ec           0
기온(°C)       0
강수량(mm)      0
풍속(m/s)      0
습도(%)        0
현지기압(hPa)    0
지면온도(°C)     0
dtype: int64


C:\Users\jcu03\AppData\Local\Temp\ipykernel_18648\2695812886.py:10: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df_interpolated = df.interpolate(method='linear', axis=0).round(1)


In [8]:
df_interpolated.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 305323 entries, 0 to 305322
Data columns (total 10 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   ymd        305323 non-null  object 
 1   code_new   305323 non-null  int64  
 2   wtemp      305323 non-null  float64
 3   ec         305323 non-null  float64
 4   기온(°C)     305323 non-null  float64
 5   강수량(mm)    305323 non-null  float64
 6   풍속(m/s)    305323 non-null  float64
 7   습도(%)      305323 non-null  float64
 8   현지기압(hPa)  305323 non-null  float64
 9   지면온도(°C)   305323 non-null  float64
dtypes: float64(8), int64(1), object(1)
memory usage: 23.3+ MB


In [10]:
# -------------------------------
# 5) 엑셀로 저장 (선택 사항)
# -------------------------------
df_interpolated.to_csv("파일이름_interpolated.csv", index=False, encoding="cp949")